# LastDance verified replay SFT
Enable GPU and Internet. Attach amnpham/zalochallenge and a private dataset containing lastdance_verified_sft_inputs.zip (or its extracted files). Train uses the verified 4,498 rows, Dev20 token loss and one final Val200 evaluation.


In [ ]:
import json
import os
from pathlib import Path
import subprocess
import sys

WORK = Path("/kaggle/working/lastdance_verified_sft")
REPO = Path("/kaggle/working/LastDance")
DATA_ROOT = Path("/kaggle/input/datasets/amnpham/zalochallenge/train+public_test")
BATCH = 8
SAVE_STEPS = 20
RUN_TAG = "full_fresh_lora"
TEACHER_DIR = WORK / "train_example"

if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/AmnO-O/LastDance.git", str(REPO)], check=True)
subprocess.run(["git", "pull", "--ff-only"], cwd=REPO, check=True)
if not (DATA_ROOT / "train" / "train.jsonl").is_file():
    raise FileNotFoundError(DATA_ROOT / "train" / "train.jsonl")
import zipfile
archives = sorted(Path("/kaggle/input").rglob("lastdance_verified_sft_inputs.zip"))
if archives:
    if len(archives) != 1:
        raise ValueError("Attach exactly one verified input archive.")
    verified = WORK / "verified_inputs"
    verified.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archives[0]) as archive:
        for name in ("teacher_ds.jsonl", "manifest.json", "train_gipformer.json"):
            (verified / name).write_bytes(archive.read(name))
else:
    datasets = sorted(Path("/kaggle/input").rglob("teacher_ds.jsonl"))
    if len(datasets) != 1:
        raise ValueError("Attach exactly one verified replay dataset.")
    verified = datasets[0].parent
paths = [verified / "train_gipformer.json"]
env = dict(os.environ, LIFEMATE_DATA_ROOT=str(DATA_ROOT), LIFEMATE_TRANSCRIPTS=str(paths[0]),
           LIFEMATE_WORK=str(WORK), PYTHONUNBUFFERED="1")

def run_module(module, *args, extra_env=None):
    subprocess.run([sys.executable, "-m", module, *args],
                   cwd=REPO, env={**env, **(extra_env or {})}, check=True)

def stage(*args, extra_env=None):
    run_module("pipeline.run_search_sft", *args, extra_env=extra_env)


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO / "requirements.txt")], check=True)
subprocess.run([sys.executable, "-c", "import torch; assert torch.cuda.is_available(); print(torch.cuda.device_count(), 'GPU(s)')"], check=True)


In [ ]:
stage("prepare", "--full-train", "--transcripts", "mixed", "--verified-dataset", str(verified))


In [ ]:
import math

manifest = json.loads((WORK / "gold_replay_manifest.json").read_text())
assert manifest["full_train"] and manifest["transcripts"] == "mixed"
assert manifest["preopen_tools"] == 12
teacher = json.loads((TEACHER_DIR / "manifest.json").read_text())
assert teacher["prompt_sha256"] == "5268fcf5470fdfb2adb3eabb0eeb857cfc060f4413b81f8d3d4974bd4ec111ed"
assert teacher["prompt_matches_inference"] is True
assert teacher["config"]["preopen_tools"] == 12
assert teacher["files"]["teacher_ds"]["rows"]
print(f"{teacher['files']['teacher_ds']['rows']} teacher rows, "
      f"{teacher['sample_coverage']} samples")
rows = manifest["examples"]["train"]["count"]
if not rows:
    raise RuntimeError("No accepted training rows.")
steps = math.ceil(rows / (BATCH * SAVE_STEPS)) * SAVE_STEPS
print(f"{rows} accepted rows, {steps} optimizer steps, batch {BATCH}")
stage("train", "--steps", str(steps), "--save-steps", str(SAVE_STEPS),
      "--batch", str(BATCH), "--learning-rate", "1e-5", "--run-tag", RUN_TAG)


In [ ]:
stage("val", "--adapter", str(WORK / RUN_TAG / "adapter"))
print((WORK / "val_report.json").read_text(encoding="utf-8"))
